In [1]:
import warnings
warnings.filterwarnings('ignore') #, category=UserWarning)
import numpy as np
from vorothreshold import voronoi_threshold_finder
from vorothreshold.read_funcs import read_voronoi_vide
import matplotlib.pyplot as plt

The standard methodology to postprocess vide voids in boxes in order to find the corresponding threshold voids, requires two quantities, which are:

- A list/array or a single value of thresholds. This should be passed in terms of the normalized density, i.e. $1+\delta$

- The path of vide directory, the one containing all the output files

In addition, the `lightcone` input variable must be set to `False`:

In [ ]:
thresholds = [0.2,0.3]
vide_path='data/box/examples/example_simulation/sim_ss1.0/sample_sim_ss1.0_z0.00_d00/'
th_voids = voronoi_threshold_finder(thresholds,lightcone=False,vide_path=vide_path,verbose=True)

The core functions called by the `voronoi_threshold_finder` class are in `Numba`. During the first call, the are compiled, so the total time appearing in the upper cell is the sum between compilation and execution. To demonstrate the speed of execution only we initialize one again the `voronoi_threshold_finder` class.

In [ ]:
th_voids = voronoi_threshold_finder(thresholds,lightcone=False,vide_path=vide_path,verbose=True)

To get threshold void properties, we use the `get_values` function from the `voronoi_threshold_finder` class. The arguments are the specific threshold value, among the values in `thresholds`, the quantity we want to obtain, and the overlapping fraction,`frac_ovlp`. This last quantity represents the maximum fractional amount of volume shared with an anther void. It is a free parameter, any value below or equal to 0.5 is fine. The corresponding output will contains voids which overlaps for a fraction of their volume minor or equal to `frac_ovlp`. The use of `frac_ovlp` is necessary to avoid double counting, as the thresholding procedure may merge adjacent basins in a unique void.

The ids of non-overlapping voids for a specific `frac_ovlp` is computed the first time that the value is passed to the `get_values` function. Alternatively, the ids can be computed by calling the `compute_overlaps` function, which accept as arguments both a single `frac_ovlp`, or multiple values passed as a list or a `numpy` array. In such a case, all the computations for all the combination `frac_ovlp` and `thresholds` values will be performed in parallel.

In [ ]:
frac_ovlp_list = [0.5,0.25,0.]

th_voids.compute_overlaps(np.array(frac_ovlp_list),verbose=False)

print('keys:',th_voids.id_out.keys())
print('keys:',th_voids.id_out[0].keys())

The following cell shows how to extract some threshold void properties with `get_values`. The list of available quantities is the following: 

- Ncells: Number of Voronoi cells contained in each void. This is an array of float as the last Voronoi cell adds as the volume fraction considered to reach the threshold value. dtype=float64, shape=(N,)
        
- ID_original_sample: IDs of the void of the original VIDE catalog used to start the thresholding process. dtype=int64, shape=(N,)
            
- id_selected: IDs of voids of the entire voronoi_threshold output that reach the threshold value and satisfying the overlaps condition. dtype=int64, shape=(N,)

- id_wrt_all: IDs of voids of the entire voronoi_threshold output that reach the threshold value and satisfying the overlaps condition. dtype=int64, shape=(N,)

- xyz: Comoving coordinates of the volume weighted barycenter. dtype=float64, shape=(N,3)

- volume: Void volumes. dtype=float64, shape=(N,)

- radius: Void effective radius. dtype=float64, shape=(N,)

- ell_eigenvalues: eigenvalues of the inertial tensor. dtype=float64, shape=(N,3)

- ell_eigenvectors: eigenvectors of the inertial tensor. dtype=float64, shape=(N,3,3)

In [5]:
xyz = dict()
radius = dict()
for thr in thresholds:
    xyz[thr] = dict()
    radius[thr] = dict()
    for frac_ovlp in frac_ovlp_list:
        xyz[thr][frac_ovlp] = th_voids.get_values(thr,'xyz',frac_ovlp)
        radius[thr][frac_ovlp] = th_voids.get_values(thr,'radius',frac_ovlp)

Example of voids with different overlapping fraction values in a slice of the simulation box. The radius of each void correspond to the effective radius. Note however that voids are not spherical.

In [ ]:
import matplotlib
fig = plt.figure(figsize=(8,8))
ax0 = plt.subplot2grid((1,1), (0,0), rowspan=1, colspan=1)

from vorothreshold.read_funcs import load_pickle_safe
Lbox = load_pickle_safe(vide_path+'/sample_info.dat')['boxLen']
progr = 0
for frac_ovlp in frac_ovlp_list:
    cc = plt.cm.tab10(progr)
    mask = xyz[thr][frac_ovlp][:,2] < 100.
    patches = []
    CC = []
    for i in range(np.sum(mask)):
        # circle centered in #RAcm[i],DECcm[i] with radius dist_ang_max[i] * 180 / np.pi
        patches.append(plt.Circle((xyz[thr][frac_ovlp][mask,0][i], xyz[thr][frac_ovlp][mask,1][i]), radius[thr][frac_ovlp][mask][i]))
        CC.append(cc)
    coll = matplotlib.collections.PatchCollection(patches,facecolors='None',linestyle='--',linewidth=0.5,edgecolor=CC)
    ax0.scatter(xyz[thr][frac_ovlp][mask,0],xyz[thr][frac_ovlp][mask,1],s=15,label='overlapping fraction $\leq$ '+str(frac_ovlp))
    ax0.add_collection(coll)
    progr += 1
ax0.axis('equal')
ax0.set_xlabel('x')
ax0.set_ylabel('y')
ax0.grid()
ax0.plot([0,Lbox,Lbox,0,0],[0,0,Lbox,Lbox,0],c='k',lw=1,ls='--')
plt.legend()

Check that the void whit the maximum number of Voronoi cells does not exceeds the maximum number of Voronoi cells allocated for each void. 

In [ ]:
max_voro_in_voids = np.max(th_voids.get_values(np.max(thresholds),'Ncells',1.))
max_voro_in_voids = int(max_voro_in_voids) + int(max_voro_in_voids%1 > 0)
print('max_num_part passed:',th_voids.max_num_part,'\nmax num of voronoi in one void:',max_voro_in_voids,
      '\nNvds with num max:',np.sum(th_voids.get_values(np.max(thresholds),'Ncells',1.)==max_voro_in_voids))

Plot of the spatial distribution of Voronoi cells belonging to threshold voids with respect to the full galaxy distribution. 

Using the function `void_contour_2D` from the `plotting_functions` module we show the approximated projected contour of voids, which is not spherical.

In [ ]:
from vorothreshold.plotting_functions import void_contour_2D
# load voronoi cell (tracer) coordinates:
VoroXYZ = read_voronoi_vide(vide_path)[2]

# set threshold id (ith) and value (thr)
ith = len(thresholds)-1
thr = thresholds[ith]

# center the plot on the larger void in the catalog
xyz_ref = th_voids.Xcm[np.argmax(th_voids.Ncells_in_void[:,ith]),ith,:] #np.mean(VoroXYZ,axis=0) # the slab center has been chose as the mean galaxy position. It can be substituted with any value

x_thick = 300. # half-plane extension in the x direction
y_thick = 300. # half-plane extension in the y direction
z_thick = 100.  # half thickness of the projection along the z direction

# PBC tracers wrt xyz_ref
xyz_trs = np.copy(VoroXYZ)
xyz_trs[(xyz_trs - xyz_ref) > Lbox/2] -= Lbox
xyz_trs[(xyz_trs - xyz_ref) < -Lbox/2] += Lbox

# select tracers in the slab
mask_trs = (xyz_trs[:,0] >= xyz_ref[0] - x_thick) & (xyz_trs[:,0] <= xyz_ref[0] + x_thick) & \
           (xyz_trs[:,1] >= xyz_ref[1] - y_thick) & (xyz_trs[:,1] <= xyz_ref[1] + y_thick) & \
           (xyz_trs[:,2] >= xyz_ref[2] - z_thick) & (xyz_trs[:,2] <= xyz_ref[2] + z_thick)

# PBC voids wrt xyz_ref
voids_xyz = np.copy(th_voids.Xcm[:,ith,:])
voids_xyz[(voids_xyz - xyz_ref) > Lbox/2] -= Lbox
voids_xyz[(voids_xyz - xyz_ref) < -Lbox/2] += Lbox

# select voids with their center in the slab
mask_voids = np.zeros(voids_xyz.shape[0],dtype=np.bool_)
mask_voids[th_voids.get_values(thr,'id_wrt_all',frac_ovlp)] = True
mask_voids &= (voids_xyz[:,0] >= xyz_ref[0] - x_thick) & (voids_xyz[:,0] <= xyz_ref[0] + x_thick) & \
             (voids_xyz[:,1] >= xyz_ref[1] - y_thick) & (voids_xyz[:,1] <= xyz_ref[1] + y_thick) & \
             (voids_xyz[:,2] >= xyz_ref[2] - z_thick) & (voids_xyz[:,2] <= xyz_ref[2] + z_thick)
voids_xyz = voids_xyz[mask_voids,:]
id_ovlp_out = np.arange(mask_voids.shape[0])[mask_voids]
id_progr = np.arange(mask_voids.shape[0])[mask_voids]

fig = plt.figure(figsize=[7,7])
cprogr = 0
for iprogr in range(np.sum(mask_voids)):
    ii = id_ovlp_out[iprogr]
    # select tracers both in voids
    ids_in_voids = th_voids.ID_voro_dict[ii][:int(th_voids.Ncells_in_void[ii,ith]+1)]

    mask_trs_loop = (xyz_trs[ids_in_voids,0] >= xyz_ref[0] - x_thick) & (xyz_trs[ids_in_voids,0] <= xyz_ref[0] + x_thick) & \
                    (xyz_trs[ids_in_voids,1] >= xyz_ref[1] - y_thick) & (xyz_trs[ids_in_voids,1] <= xyz_ref[1] + y_thick) & \
                    (xyz_trs[ids_in_voids,2] >= xyz_ref[2] - z_thick) & (xyz_trs[ids_in_voids,2] <= xyz_ref[2] + z_thick)
    if int(th_voids.Ncells_in_void[ii,ith]+1) < 10:
        continue
    mask_trs[ids_in_voids] = False # remove from the general selection tracers that are both in voids and in the slab
    
    # plot void contour and tracers within it with the same color
    cprogr += (cprogr == 7) # skip grey
    cprogr += (cprogr == 5) # skip brown

    cc = plt.cm.tab10(cprogr)
    scat = plt.scatter(xyz_trs[ids_in_voids,0],xyz_trs[ids_in_voids,1],s=1,color=cc)
    plt.scatter(voids_xyz[iprogr,0],voids_xyz[iprogr,1],c=scat.get_facecolors(),alpha=1)
     
    plt.plot(*void_contour_2D(xyz_trs[ids_in_voids,:]),lw=5,alpha=0.4,c=scat.get_facecolors())
    cprogr = (cprogr+1)%10

# plot tracers within the slab but not belonging to selected voids in black
plt.scatter([],[],c='k',alpha=0.5,label='Void barycenter')
plt.plot([],[],lw=5,alpha=0.2,c='k',label='Void 2D contour')
plt.scatter(xyz_trs[mask_trs,0],xyz_trs[mask_trs,1],s=0.5,c='k',alpha=0.2,label='Tracers')
plt.legend(loc='upper right')
plt.axis('equal')
plt.xlabel('Mpc/$h$')
plt.ylabel('Mpc/$h$')
plt.grid()
fig.tight_layout()
plt.xlim(xyz_ref[0] - x_thick*0.9,xyz_ref[0] + x_thick*0.9)
plt.ylim(xyz_ref[1] - y_thick,xyz_ref[1] + y_thick)